In [ ]:
import re
import json
from pathlib import Path
from urllib.parse import urlparse, parse_qs, urlencode

# =========================
# 1) PASTE DATA HERE
# =========================
HANOI_CATEGORIES_RAW = r'''Hà nội , tất cả các quận
https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ha-noi-l1?sort=up_top&type_keyword=1&sba=1&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-kinh-doanh-ban-hang-tai-ha-noi-l1cr1?sort=up_top&type_keyword=0&sba=1&category_family=r1&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-marketing-pr-quang-cao-tai-ha-noi-l1cr92?sort=up_top&type_keyword=0&sba=1&category_family=r92&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-cham-soc-khach-hang-customer-service-van-hanh-tai-ha-noi-l1cr158?sort=up_top&type_keyword=0&sba=1&category_family=r158&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-nhan-su-hanh-chinh-phap-che-tai-ha-noi-l1cr177?sort=up_top&type_keyword=0&sba=1&category_family=r177&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-cong-nghe-thong-tin-tai-ha-noi-l1cr257?sort=up_top&type_keyword=0&sba=1&category_family=r257&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-lao-dong-pho-thong-tai-ha-noi-l1cr1042?sort=up_top&type_keyword=0&sba=1&category_family=r1042&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-tai-chinh-ngan-hang-bao-hiem-tai-ha-noi-l1cr206?sort=up_top&type_keyword=0&sba=1&category_family=r206&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-bat-dong-san-tai-ha-noi-l1cr333?sort=up_top&type_keyword=0&sba=1&category_family=r333&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-xay-dung-tai-ha-noi-l1cr1080?sort=up_top&type_keyword=0&sba=1&category_family=r1080&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-ke-toan-kiem-toan-thue-tai-ha-noi-l1cr392?sort=up_top&type_keyword=0&sba=1&category_family=r392&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-san-xuat-tai-ha-noi-l1cr417?sort=up_top&type_keyword=0&sba=1&category_family=r417&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-giao-duc-dao-tao-tai-ha-noi-l1cr477?sort=up_top&type_keyword=0&sba=1&category_family=r477&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-ban-le-dich-vu-doi-song-tai-ha-noi-l1cr544?sort=up_top&type_keyword=0&sba=1&category_family=r544&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-phim-va-truyen-hinh-bao-chi-xuat-ban-tai-ha-noi-l1cr612?sort=up_top&type_keyword=0&sba=1&category_family=r612&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-dien-dien-tu-vien-thong-tai-ha-noi-l1cr644?sort=up_top&type_keyword=0&sba=1&category_family=r644&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-logistics-thu-mua-kho-van-tai-tai-ha-noi-l1cr711?sort=up_top&type_keyword=0&sba=1&category_family=r711&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-tu-van-chuyen-mon-tai-ha-noi-l1cr750?sort=up_top&type_keyword=0&sba=1&category_family=r750&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-duoc-y-te-suc-khoe-cong-nghe-sinh-hoc-tai-ha-noi-l1cr781?sort=up_top&type_keyword=0&sba=1&category_family=r781&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-thiet-ke-tai-ha-noi-l1cr826?sort=up_top&type_keyword=0&sba=1&category_family=r826&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-nha-hang-khach-san-du-lich-tai-ha-noi-l1cr857?sort=up_top&type_keyword=0&sba=1&category_family=r857&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-nang-luong-moi-truong-nong-nghiep-tai-ha-noi-l1cr883?sort=up_top&type_keyword=0&sba=1&category_family=r883&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-tai-xe-tai-ha-noi-l1cr1010?sort=up_top&type_keyword=0&sba=1&category_family=r1010&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-bien-phien-dich-tai-ha-noi-l1cr1013?sort=up_top&type_keyword=0&sba=1&category_family=r1013&locations=l1&saturday_status=0&location_version_query=2

https://www.topcv.vn/tim-viec-lam-luat-tai-ha-noi-l1cr1014?sort=up_top&type_keyword=0&sba=1&category_family=r1014&locations=l1&saturday_status=0&location_version_query=2
'''

LOCATIONS_RAW = r'''phường ba đinh
https://www.topcv.vn/tim-viec-lam-tai-phuong-ba-dinh-l1d20058lv2?type_keyword=0&sba=1&locations=l1d20058&saturday_status=0&location_version_query=2

phường bạch mai 
https://www.topcv.vn/tim-viec-lam-tai-phuong-bach-mai-l1d20095lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20095&saturday_status=0&location_version_query=2

phường bồ đề 
https://www.topcv.vn/tim-viec-lam-tai-phuong-bo-de-l1d20111lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20111&saturday_status=0&location_version_query=2

phường chương mỹ
https://www.topcv.vn/tim-viec-lam-tai-phuong-chuong-my-l1d20134lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20134&saturday_status=0&location_version_query=2

phường cầu giấy 
https://www.topcv.vn/tim-viec-lam-tai-phuong-cau-giay-l1d20143lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20143&saturday_status=0&location_version_query=2

phường cửa nam
https://www.topcv.vn/tim-viec-lam-tai-phuong-cua-nam-l1d20151lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20151&saturday_status=0&location_version_query=2

phường dương nội

https://www.topcv.vn/tim-viec-lam-tai-phuong-duong-noi-l1d20162lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20162&saturday_status=0&location_version_query=2

phường giảng võ

https://www.topcv.vn/tim-viec-lam-tai-phuong-giang-vo-l1d20169lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20169&saturday_status=0&location_version_query=2

phường hai bà trưng 
https://www.topcv.vn/tim-viec-lam-tai-phuong-hai-ba-trung-l1d20172lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20172&saturday_status=0&location_version_query=2

phường hoàn kiếm

https://www.topcv.vn/tim-viec-lam-tai-phuong-hoan-kiem-l1d20182lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20182&saturday_status=0&location_version_query=2

phường hoàng liệt 
https://www.topcv.vn/tim-viec-lam-tai-phuong-hoang-liet-l1d20183lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20183&saturday_status=0&location_version_query=2

phường hoàng mai 

https://www.topcv.vn/tim-viec-lam-tai-phuong-hoang-mai-l1d20185lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20185&saturday_status=0&location_version_query=2

phường hà đông
https://www.topcv.vn/tim-viec-lam-tai-phuong-ha-dong-l1d20197lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20197&saturday_status=0&location_version_query=2

phường hồng hà
https://www.topcv.vn/tim-viec-lam-tai-phuong-hong-ha-l1d20235lv2?type_keyword=0&sba=1&locations=l1d20235&saturday_status=0&location_version_query=2

phường khương đình

https://www.topcv.vn/tim-viec-lam-tai-phuong-khuong-dinh-l1d20244lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20244&saturday_status=0&location_version_query=2

phường kim liên 
https://www.topcv.vn/tim-viec-lam-tai-phuong-kim-lien-l1d20246lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20246&saturday_status=0&location_version_query=2


phường kiến hưng

https://www.topcv.vn/tim-viec-lam-tai-phuong-kien-hung-l1d20253lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20253&saturday_status=0&location_version_query=2

phường long biên 
https://www.topcv.vn/tim-viec-lam-tai-phuong-long-bien-l1d20266lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20266&saturday_status=0&location_version_query=2

phường láng 
https://www.topcv.vn/tim-viec-lam-tai-phuong-lang-l1d20286lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20286&saturday_status=0&location_version_query=2

phường lĩnh nam

https://www.topcv.vn/tim-viec-lam-tai-phuong-linh-nam-l1d20296lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20296&saturday_status=0&location_version_query=2


phường nghĩa đô
https://www.topcv.vn/tim-viec-lam-tai-phuong-nghia-do-l1d20340lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20340&saturday_status=0&location_version_query=2

phường ngọc hà
https://www.topcv.vn/tim-viec-lam-tai-phuong-ngoc-ha-l1d20350lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20350&saturday_status=0&location_version_query=2

phường phú diễn 
https://www.topcv.vn/tim-viec-lam-tai-phuong-phu-dien-l1d20381lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20381&saturday_status=0&location_version_query=2

phường phú lương 
https://www.topcv.vn/tim-viec-lam-tai-phuong-phu-luong-l1d20384lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20384&saturday_status=0&location_version_query=2

phường phú thượng
https://www.topcv.vn/tim-viec-lam-tai-phuong-phu-thuong-l1d20390lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20390&saturday_status=0&location_version_query=2

phường phúc lợi
https://www.topcv.vn/tim-viec-lam-tai-phuong-phuc-loi-l1d20400lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20400&saturday_status=0&location_version_query=2

phường phương liệt 
https://www.topcv.vn/tim-viec-lam-tai-phuong-phuong-liet-l1d20404lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20404&saturday_status=0&location_version_query=2

phường sơn tây 
https://www.topcv.vn/tim-viec-lam-tai-phuong-son-tay-l1d20447lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20447&saturday_status=0&location_version_query=2

phường thanh liệt 
https://www.topcv.vn/tim-viec-lam-tai-phuong-thanh-liet-l1d20463lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20463&saturday_status=0&location_version_query=2

phường thanh xuân
https://www.topcv.vn/tim-viec-lam-tai-phuong-thanh-xuan-l1d20466lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20466&saturday_status=0&location_version_query=2

phường thượng cát
https://www.topcv.vn/tim-viec-lam-tai-phuong-thuong-cat-l1d20484lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20484&saturday_status=0&location_version_query=2

phường tây hồ 
https://www.topcv.vn/tim-viec-lam-tai-phuong-tay-ho-l1d20568lv2?type_keyword=0&sba=1&locations=l1d20568&saturday_status=0&location_version_query=2

phường tây mỗ 
https://www.topcv.vn/tim-viec-lam-tai-phuong-tay-mo-l1d20569lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20569&saturday_status=0&location_version_query=2

phường tây tựu https://www.topcv.vn/tim-viec-lam-tai-phuong-tay-tuu-l1d20573lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20573&saturday_status=0&location_version_query=2

phường tùng thiện 
https://www.topcv.vn/tim-viec-lam-tai-phuong-tung-thien-l1d20577lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20577&saturday_status=0&location_version_query=2

phường tương mai

https://www.topcv.vn/tim-viec-lam-tai-phuong-tuong-mai-l1d20580lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20580&saturday_status=0&location_version_query=2

phường từ liêm
https://www.topcv.vn/tim-viec-lam-tai-phuong-tu-liem-l1d20583lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20583&saturday_status=0&location_version_query=2

phường việt hưng 
https://www.topcv.vn/tim-viec-lam-tai-phuong-viet-hung-l1d20591lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20591&saturday_status=0&location_version_query=2

phường văn miếu quốc tử giám

https://www.topcv.vn/tim-viec-lam-tai-phuong-van-mieu-quoc-tu-giam-l1d20600lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20600&saturday_status=0&location_version_query=2

phường vĩnh hưng
https://www.topcv.vn/tim-viec-lam-tai-phuong-vinh-hung-l1d20603lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20603&saturday_status=0&location_version_query=2

phường vĩnh tuy
https://www.topcv.vn/tim-viec-lam-tai-phuong-vinh-tuy-l1d20608lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20608&saturday_status=0&location_version_query=2

phường xuân phương 
https://www.topcv.vn/tim-viec-lam-tai-phuong-xuan-phuong-l1d20626lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20626&saturday_status=0&location_version_query=2

phường xuân đỉnh
https://www.topcv.vn/tim-viec-lam-tai-phuong-xuan-dinh-l1d20629lv2?type_keyword=0&sba=1&locations=l1d20629&saturday_status=0&location_version_query=2

phường yên hòa
https://www.topcv.vn/tim-viec-lam-tai-phuong-yen-hoa-l1d20632lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20632&saturday_status=0&location_version_query=2

phường yên nghĩa 
https://www.topcv.vn/tim-viec-lam-tai-phuong-yen-nghia-l1d20633lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20633&saturday_status=0&location_version_query=2

phường yên sở
https://www.topcv.vn/tim-viec-lam-tai-phuong-yen-so-l1d20635lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20635&saturday_status=0&location_version_query=2

phường ô chợ dừa
https://www.topcv.vn/tim-viec-lam-tai-phuong-o-cho-dua-l1d20641lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20641&saturday_status=0&location_version_query=2

phường đông ngạc
https://www.topcv.vn/tim-viec-lam-tai-phuong-dong-ngac-l1d20663lv2?sort=up_top&type_keyword=1&sba=1&locations=l1d20663&saturday_status=0&location_version_query=2

phường đại mỗ
https://www.topcv.vn/tim-viec-lam-tai-phuong-dai-mo-l1d20672lv2?type_keyword=0&sba=1&locations=l1d20672&saturday_status=0&location_version_query=2

phường định công 
https://www.topcv.vn/tim-viec-lam-tai-phuong-dinh-cong-l1d20675lv2?type_keyword=0&sba=1&locations=l1d20675&saturday_status=0&location_version_query=2

phường đống đa
https://www.topcv.vn/tim-viec-lam-tai-phuong-dong-da-l1d20676lv2?type_keyword=0&sba=1&locations=l1d20676&saturday_status=0&location_version_query=2

xã an khánh
https://www.topcv.vn/tim-viec-lam-tai-xa-an-khanh-l1d20717lv2?type_keyword=0&sba=1&locations=l1d20717&saturday_status=0&location_version_query=2

xã ba vì
https://www.topcv.vn/tim-viec-lam-tai-xa-ba-vi-l1d20773lv2?type_keyword=0&sba=1&locations=l1d20773&saturday_status=0&location_version_query=2

xã bát tràng
https://www.topcv.vn/tim-viec-lam-tai-xa-bat-trang-l1d20799lv2?type_keyword=0&sba=1&locations=l1d20799&saturday_status=0&location_version_query=2

xã bình minh
https://www.topcv.vn/tim-viec-lam-tai-xa-binh-minh-l1d20833lv2?type_keyword=0&sba=1&locations=l1d20833&saturday_status=0&location_version_query=2

xã bất bạt
https://www.topcv.vn/tim-viec-lam-tai-xa-bat-bat-l1d20897lv2?type_keyword=0&sba=1&locations=l1d20897&saturday_status=0&location_version_query=2

xã chuyên mỹ
https://www.topcv.vn/tim-viec-lam-tai-xa-chuyen-my-l1d20975lv2?type_keyword=0&sba=1&locations=l1d20975&saturday_status=0&location_version_query=2

xã chương dương
https://www.topcv.vn/tim-viec-lam-tai-xa-chuong-duong-l1d21011lv2?type_keyword=0&sba=1&locations=l1d21011&saturday_status=0&location_version_query=2

xã cổ đô 
https://www.topcv.vn/tim-viec-lam-tai-xa-co-do-l1d21093lv2?type_keyword=0&sba=1&locations=l1d21093&saturday_status=0&location_version_query=2

xã dân hòa

https://www.topcv.vn/tim-viec-lam-tai-xa-dan-hoa-l1d21120lv2?type_keyword=0&sba=1&locations=l1d21120&saturday_status=0&location_version_query=2

xã dương hòa
https://www.topcv.vn/tim-viec-lam-tai-xa-duong-hoa-l1d21124lv2?type_keyword=0&sba=1&locations=l1d21124&saturday_status=0&location_version_query=2

xã gia lâm
https://www.topcv.vn/tim-viec-lam-tai-xa-gia-lam-l1d21168lv2?type_keyword=0&sba=1&locations=l1d21168&saturday_status=0&location_version_query=2

xã hoài đức
https://www.topcv.vn/tim-viec-lam-tai-xa-hoai-duc-l1d21221lv2?type_keyword=0&sba=1&locations=l1d21221&saturday_status=0&location_version_query=2

xã hát môn 
https://www.topcv.vn/tim-viec-lam-tai-xa-hat-mon-l1d21265lv2?type_keyword=0&sba=1&locations=l1d21265&saturday_status=0&location_version_query=2

xã hòa lạc
https://www.topcv.vn/tim-viec-lam-tai-xa-hoa-lac-l1d21280lv2?type_keyword=0&sba=1&locations=l1d21280&saturday_status=0&location_version_query=2

xã hòa phú 
https://www.topcv.vn/tim-viec-lam-tai-xa-hoa-phu-l1d21285lv2?type_keyword=0&sba=1&locations=l1d21285&saturday_status=0&location_version_query=2

xã hòa xá
https://www.topcv.vn/tim-viec-lam-tai-xa-hoa-xa-l1d21296lv2?type_keyword=0&sba=1&locations=l1d21296&saturday_status=0&location_version_query=2

xã hưng đạo
https://www.topcv.vn/tim-viec-lam-tai-xa-hung-dao-l1d21330lv2?type_keyword=0&sba=1&locations=l1d21330&saturday_status=0&location_version_query=2

xã hương sơn
https://www.topcv.vn/tim-viec-lam-tai-xa-huong-son-l1d21336lv2?type_keyword=0&sba=1&locations=l1d21336&saturday_status=0&location_version_query=2

xã hạ bằng 
https://www.topcv.vn/tim-viec-lam-tai-xa-ha-bang-l1d21343lv2?type_keyword=0&sba=1&locations=l1d21343&saturday_status=0&location_version_query=2

xã hồng sơn
https://www.topcv.vn/tim-viec-lam-tai-xa-hong-son-l1d21388lv2?type_keyword=0&sba=1&locations=l1d21388&saturday_status=0&location_version_query=2

xã hồng vân
https://www.topcv.vn/tim-viec-lam-tai-xa-hong-van-l1d21392lv2?type_keyword=0&sba=1&locations=l1d21392&saturday_status=0&location_version_query=2

xã kim anh 
https://www.topcv.vn/tim-viec-lam-tai-xa-kim-anh-l1d21480lv2?type_keyword=0&sba=1&locations=l1d21480&saturday_status=0&location_version_query=2

xã kiều phú 
https://www.topcv.vn/tim-viec-lam-tai-xa-kieu-phu-l1d21511lv2?type_keyword=0&sba=1&locations=l1d21511&saturday_status=0&location_version_query=2

xã liên minh

https://www.topcv.vn/tim-viec-lam-tai-xa-lien-minh-l1d21566lv2?type_keyword=0&sba=1&locations=l1d21566&saturday_status=0&location_version_query=2

xã minh châu 
https://www.topcv.vn/tim-viec-lam-tai-xa-minh-chau-l1d21686lv2?type_keyword=0&sba=1&locations=l1d21686&saturday_status=0&location_version_query=2

xã mê linh
https://www.topcv.vn/tim-viec-lam-tai-xa-me-linh-l1d21709lv2?type_keyword=0&sba=1&locations=l1d21709&saturday_status=0&location_version_query=2

xã mỹ đức
https://www.topcv.vn/tim-viec-lam-tai-xa-my-duc-l1d21804lv2?type_keyword=0&sba=1&locations=l1d21804&saturday_status=0&location_version_query=2

xã nam phù 
https://www.topcv.vn/tim-viec-lam-tai-xa-nam-phu-l1d21836lv2?type_keyword=0&sba=1&locations=l1d21836&saturday_status=0&location_version_query=2
xã ngọc hồi
https://www.topcv.vn/tim-viec-lam-tai-xa-ngoc-hoi-l1d21912lv2?type_keyword=0&sba=1&locations=l1d21912&saturday_status=0&location_version_query=2

xã nội bài 
https://www.topcv.vn/tim-viec-lam-tai-xa-noi-bai-l1d21998lv2?type_keyword=0&sba=1&locations=l1d21998&saturday_status=0&location_version_query=2

xã phù đổng 
https://www.topcv.vn/tim-viec-lam-tai-xa-phu-dong-l1d22038lv2?type_keyword=0&sba=1&locations=l1d22038&saturday_status=0&location_version_query=2

xã phú cát
https://www.topcv.vn/tim-viec-lam-tai-xa-phu-cat-l1d22042lv2?type_keyword=0&sba=1&locations=l1d22042&saturday_status=0&location_version_query=2

xã phú nghĩa 
https://www.topcv.vn/tim-viec-lam-tai-xa-phu-nghia-l1d22070lv2?type_keyword=0&sba=1&locations=l1d22070&saturday_status=0&location_version_query=2

xã phú xuyên 
https://www.topcv.vn/tim-viec-lam-tai-xa-phu-xuyen-l1d22095lv2?type_keyword=0&sba=1&locations=l1d22095&saturday_status=0&location_version_query=2

xã phúc lộc
https://www.topcv.vn/tim-viec-lam-tai-xa-phuc-loc-l1d22102lv2?type_keyword=0&sba=1&locations=l1d22102&saturday_status=0&location_version_query=2

xã phúc sơn 
https://www.topcv.vn/tim-viec-lam-tai-xa-phuc-son-l1d22104lv2?type_keyword=0&sba=1&locations=l1d22104&saturday_status=0&location_version_query=2

xã phúc thịnh
https://www.topcv.vn/tim-viec-lam-tai-xa-phuc-thinh-l1d22105lv2?type_keyword=0&sba=1&locations=l1d22105&saturday_status=0&location_version_query=2

xã phúc thọ
https://www.topcv.vn/tim-viec-lam-tai-xa-phuc-tho-l1d22106lv2?type_keyword=0&sba=1&locations=l1d22106&saturday_status=0&location_version_query=2

xã phượng dực
https://www.topcv.vn/tim-viec-lam-tai-xa-phuong-duc-l1d22135lv2?type_keyword=0&sba=1&locations=l1d22135&saturday_status=0&location_version_query=2

xã quang minh 
https://www.topcv.vn/tim-viec-lam-tai-xa-quang-minh-l1d22169lv2?type_keyword=0&sba=1&locations=l1d22169&saturday_status=0&location_version_query=2

xã quảng bị
https://www.topcv.vn/tim-viec-lam-tai-xa-quang-bi-l1d22189lv2?type_keyword=0&sba=1&locations=l1d22189&saturday_status=0&location_version_query=2

xã quảng oai
https://www.topcv.vn/tim-viec-lam-tai-xa-quang-oai-l1d22203lv2?type_keyword=0&sba=1&locations=l1d22203&saturday_status=0&location_version_query=2

xã quốc oai
https://www.topcv.vn/tim-viec-lam-tai-xa-quoc-oai-l1d22222lv2?type_keyword=0&sba=1&locations=l1d22222&saturday_status=0&location_version_query=2

xã suối hai 
https://www.topcv.vn/tim-viec-lam-tai-xa-suoi-hai-l1d22258lv2?type_keyword=0&sba=1&locations=l1d22258&saturday_status=0&location_version_query=2

xã sóc sơn
https://www.topcv.vn/tim-viec-lam-tai-xa-soc-son-l1d22271lv2?type_keyword=0&sba=1&locations=l1d22271&saturday_status=0&location_version_query=2

xã sơn đồng 
https://www.topcv.vn/tim-viec-lam-tai-xa-son-dong-l1d22315lv2?type_keyword=0&sba=1&locations=l1d22315&saturday_status=0&location_version_query=2

xã tam hưng
https://www.topcv.vn/tim-viec-lam-tai-xa-tam-hung-l1d22328lv2?type_keyword=0&sba=1&locations=l1d22328&saturday_status=0&location_version_query=2

xã thanh oai
https://www.topcv.vn/tim-viec-lam-tai-xa-thanh-oai-l1d22364lv2?type_keyword=0&sba=1&locations=l1d22364&saturday_status=0&location_version_query=2

xã thanh trì
https://www.topcv.vn/tim-viec-lam-tai-xa-thanh-tri-l1d22373lv2?type_keyword=0&sba=1&locations=l1d22373&saturday_status=0&location_version_query=2

xã thiên lộc
https://www.topcv.vn/tim-viec-lam-tai-xa-thien-loc-l1d22377lv2?type_keyword=0&sba=1&locations=l1d22377&saturday_status=0&location_version_query=2

xã thuận an 
https://www.topcv.vn/tim-viec-lam-tai-xa-thuan-an-l1d22398lv2?type_keyword=0&sba=1&locations=l1d22398&saturday_status=0&location_version_query=2

xã thư lâm 
https://www.topcv.vn/tim-viec-lam-tai-xa-thu-lam-l1d22434lv2?type_keyword=0&sba=1&locations=l1d22434&saturday_status=0&location_version_query=2

xã thường tín 
https://www.topcv.vn/tim-viec-lam-tai-xa-thuong-tin-l1d22439lv2?type_keyword=0&sba=1&locations=l1d22439&saturday_status=0&location_version_query=2

xã thượng phúc 
https://www.topcv.vn/tim-viec-lam-tai-xa-thuong-phuc-l1d22449lv2?type_keyword=0&sba=1&locations=l1d22449&saturday_status=0&location_version_query=2

xã thạch thất 
https://www.topcv.vn/tim-viec-lam-tai-xa-thach-that-l1d22462lv2?type_keyword=0&sba=1&locations=l1d22462&saturday_status=0&location_version_query=2

xã tiến thắng 
https://www.topcv.vn/tim-viec-lam-tai-xa-tien-thang-l1d22537lv2?type_keyword=0&sba=1&locations=l1d22537&saturday_status=0&location_version_query=2

xã trung giã
https://www.topcv.vn/tim-viec-lam-tai-xa-trung-gia-l1d22556lv2?type_keyword=0&sba=1&locations=l1d22556&saturday_status=0&location_version_query=2

xã trần phú
https://www.topcv.vn/tim-viec-lam-tai-xa-tran-phu-l1d22622lv2?type_keyword=0&sba=1&locations=l1d22622&saturday_status=0&location_version_query=2

xã tây phương
https://www.topcv.vn/tim-viec-lam-tai-xa-tay-phuong-l1d22803lv2?type_keyword=0&sba=1&locations=l1d22803&saturday_status=0&location_version_query=2

xã vân đình 
https://www.topcv.vn/tim-viec-lam-tai-xa-van-dinh-l1d22872lv2?type_keyword=0&sba=1&locations=l1d22872&saturday_status=0&location_version_query=2

xã vĩnh thanh 
https://www.topcv.vn/tim-viec-lam-tai-xa-vinh-thanh-l1d22930lv2?type_keyword=0&sba=1&locations=l1d22930&saturday_status=0&location_version_query=2

xã vật lại 
https://www.topcv.vn/tim-viec-lam-tai-xa-vat-lai-l1d22978lv2?type_keyword=0&sba=1&locations=l1d22978&saturday_status=0&location_version_query=2

xã xuân mai
https://www.topcv.vn/tim-viec-lam-tai-xa-xuan-mai-l1d23014lv2?type_keyword=0&sba=1&locations=l1d23014&saturday_status=0&location_version_query=2

xã yên bài
https://www.topcv.vn/tim-viec-lam-tai-xa-yen-bai-l1d23045lv2?type_keyword=0&sba=1&locations=l1d23045&saturday_status=0&location_version_query=2

xã yên lãng 
https://www.topcv.vn/tim-viec-lam-tai-xa-yen-lang-l1d23059lv2?type_keyword=0&sba=1&locations=l1d23059&saturday_status=0&location_version_query=2

xã yên xuân 
https://www.topcv.vn/tim-viec-lam-tai-xa-yen-xuan-l1d23096lv2?type_keyword=0&sba=1&locations=l1d23096&saturday_status=0&location_version_query=2

xã ô diên 
https://www.topcv.vn/tim-viec-lam-tai-xa-o-dien-l1d23107lv2?type_keyword=0&sba=1&locations=l1d23107&saturday_status=0&location_version_query=2

xã đa phúc
https://www.topcv.vn/tim-viec-lam-tai-xa-da-phuc-l1d23112lv2?type_keyword=0&sba=1&locations=l1d23112&saturday_status=0&location_version_query=2

xã đan phượng
https://www.topcv.vn/tim-viec-lam-tai-xa-dan-phuong-l1d23125lv2?type_keyword=0&sba=1&locations=l1d23125&saturday_status=0&location_version_query=2

xã đoài phương
https://www.topcv.vn/tim-viec-lam-tai-xa-doai-phuong-l1d23138lv2?type_keyword=0&sba=1&locations=l1d23138&saturday_status=0&location_version_query=2

xã đông anh
https://www.topcv.vn/tim-viec-lam-tai-xa-dong-anh-l1d23152lv2?type_keyword=0&sba=1&locations=l1d23152&saturday_status=0&location_version_query=2

xã đại thanh
https://www.topcv.vn/tim-viec-lam-tai-xa-dai-thanh-l1d23226lv2?type_keyword=0&sba=1&locations=l1d23226&saturday_status=0&location_version_query=2

xã đại xuyên
https://www.topcv.vn/tim-viec-lam-tai-xa-dai-xuyen-l1d23228lv2?type_keyword=0&sba=1&locations=l1d23228&saturday_status=0&location_version_query=2

xã ứng hòa
https://www.topcv.vn/tim-viec-lam-tai-xa-ung-hoa-l1d23305lv2?type_keyword=0&sba=1&locations=l1d23305&saturday_status=0&location_version_query=2

xã ứng thiên 
https://www.topcv.vn/tim-viec-lam-tai-xa-ung-thien-l1d23306lv2?type_keyword=0&sba=1&locations=l1d23306&saturday_status=0&location_version_query=2
'''

# =========================
# 2) PARSER
# =========================
URL_RE = re.compile(r"https?://\S+")


def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())


def parse_category_templates(raw: str):
    lines = [normalize_text(x) for x in raw.splitlines() if normalize_text(x)]
    urls = [x for x in lines if x.startswith("http")]
    if not urls:
        raise ValueError("Không tìm thấy URL danh mục Hà Nội.")
    return urls


def parse_locations(raw: str):
    lines = [x.rstrip() for x in raw.splitlines()]
    items = []
    current_name = None

    for line in lines:
        line = normalize_text(line)
        if not line:
            continue
        if line.startswith("#"):
            continue

        m = URL_RE.search(line)
        if m:
            url = m.group(0)
            if current_name is None:
                current_name = ""
            items.append({"name": current_name, "url": url})
            current_name = None
        else:
            current_name = line

    if not items:
        raise ValueError("Không tìm thấy URL xã/phường.")
    return items


# =========================
# 3) URL BUILDER
# =========================
def split_path_token(location_url: str):
    """
    Ví dụ:
    /tim-viec-lam-tai-phuong-ba-dinh-l1d20058lv2
    -> location_slug = phuong-ba-dinh
    -> location_token = l1d20058lv2
    -> location_id = l1d20058
    """
    path = urlparse(location_url).path
    m = re.search(r"/tim-viec-lam-tai-(.+)-(l\d+d\d+lv\d+)$", path)
    if not m:
        raise ValueError(f"Không parse được location path: {location_url}")
    location_slug = m.group(1)
    location_token = m.group(2)
    location_id_match = re.search(r"(l\d+d\d+)", location_token)
    if not location_id_match:
        raise ValueError(f"Không parse được location id: {location_url}")
    location_id = location_id_match.group(1)
    return location_slug, location_token, location_id


def parse_category_template(category_url: str):
    """
    Ví dụ:
    /tim-viec-lam-kinh-doanh-ban-hang-tai-ha-noi-l1cr1
    -> category_slug = kinh-doanh-ban-hang
    -> category_id = 1
    -> query giữ nguyên, chỉ thay locations=l1 -> locations=<location_id>
    """
    parsed = urlparse(category_url)
    path = parsed.path

    # URL all-jobs của Hà Nội không có category_family
    if "category_family=" not in parsed.query:
        return {
            "kind": "all_jobs",
            "original_url": category_url,
            "query": parse_qs(parsed.query),
        }

    m = re.search(r"/tim-viec-lam-(.+)-tai-ha-noi-l1cr(\d+)$", path)
    if not m:
        raise ValueError(f"Không parse được category template: {category_url}")

    category_slug = m.group(1)
    category_id = m.group(2)
    query = parse_qs(parsed.query)

    return {
        "kind": "category",
        "original_url": category_url,
        "category_slug": category_slug,
        "category_id": category_id,
        "query": query,
    }


def build_url_for_location(category_template: dict, location_url: str):
    location_slug, location_token, location_id = split_path_token(location_url)

    if category_template["kind"] == "all_jobs":
        parsed_loc = urlparse(location_url)
        query = parse_qs(parsed_loc.query)
        # chuẩn hóa theo mẫu Hà Nội tổng
        query["sort"] = ["up_top"]
        query["type_keyword"] = ["1"]
        query["sba"] = ["1"]
        query["locations"] = [location_id]
        query["saturday_status"] = ["0"]
        query["location_version_query"] = ["2"]
        return f"https://www.topcv.vn{parsed_loc.path}?{urlencode(query, doseq=True)}"

    category_slug = category_template["category_slug"]
    category_id = category_template["category_id"]

    # Quy tắc build theo đúng pattern bạn đang dùng:
    # tim-viec-lam-<category>-tai-ha-noi-l1cr<ID>
    # -> tim-viec-lam-<category>-tai-<location_slug>-<location_token>cr<ID>
    new_path = f"/tim-viec-lam-{category_slug}-tai-{location_slug}-{location_token}cr{category_id}"

    query = dict(category_template["query"])
    query["sort"] = ["up_top"]
    query["type_keyword"] = ["0"]
    query["sba"] = ["1"]
    query["locations"] = [location_id]
    query["category_family"] = [f"r{category_id}"]
    query["saturday_status"] = ["0"]
    query["location_version_query"] = ["2"]

    return f"https://www.topcv.vn{new_path}?{urlencode(query, doseq=True)}"


# =========================
# 4) GENERATOR
# =========================
def generate_dataset(hanoi_categories_raw: str, locations_raw: str):
    category_urls = parse_category_templates(hanoi_categories_raw)
    locations = parse_locations(locations_raw)
    category_templates = [parse_category_template(url) for url in category_urls]

    dataset = []
    for loc in locations:
        loc_block = {
            "name": loc["name"],
            "base_url": loc["url"],
            "urls": [],
        }
        for tpl in category_templates:
            loc_block["urls"].append(build_url_for_location(tpl, loc["url"]))
        dataset.append(loc_block)
    return dataset


def export_txt(dataset, file_path="topcv_hanoi_locations_full.txt"):
    out = []
    for item in dataset:
        out.append(item["name"])
        for url in item["urls"]:
            out.append(url)
        out.append("")
    Path(file_path).write_text("\n".join(out), encoding="utf-8")
    return file_path


def export_json(dataset, file_path="topcv_hanoi_locations_full.json"):
    Path(file_path).write_text(
        json.dumps(dataset, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return file_path


if __name__ == "__main__":
    data = generate_dataset(HANOI_CATEGORIES_RAW, LOCATIONS_RAW)

    txt_file = export_txt(data)
    json_file = export_json(data)

    print(f"Đã tạo: {txt_file}")
    print(f"Đã tạo: {json_file}")
    print(f"Tổng xã/phường: {len(data)}")
    print(f"Số URL mỗi xã/phường: {len(data[0]['urls']) if data else 0}")

    # In thử block đầu tiên
    if data:
        print("\n===== SAMPLE =====")
        print(data[0]["name"])
        for u in data[0]["urls"][:5]:
            print(u)


Đã tạo: topcv_hanoi_locations_full.txt
Đã tạo: topcv_hanoi_locations_full.json
Tổng xã/phường: 126
Số URL mỗi xã/phường: 25

===== SAMPLE =====
phường ba đinh
https://www.topcv.vn/tim-viec-lam-tai-phuong-ba-dinh-l1d20058lv2?type_keyword=1&sba=1&locations=l1d20058&saturday_status=0&location_version_query=2&sort=up_top
https://www.topcv.vn/tim-viec-lam-kinh-doanh-ban-hang-tai-phuong-ba-dinh-l1d20058lv2cr1?sort=up_top&type_keyword=0&sba=1&category_family=r1&locations=l1d20058&saturday_status=0&location_version_query=2
https://www.topcv.vn/tim-viec-lam-marketing-pr-quang-cao-tai-phuong-ba-dinh-l1d20058lv2cr92?sort=up_top&type_keyword=0&sba=1&category_family=r92&locations=l1d20058&saturday_status=0&location_version_query=2
https://www.topcv.vn/tim-viec-lam-cham-soc-khach-hang-customer-service-van-hanh-tai-phuong-ba-dinh-l1d20058lv2cr158?sort=up_top&type_keyword=0&sba=1&category_family=r158&locations=l1d20058&saturday_status=0&location_version_query=2
https://www.topcv.vn/tim-viec-lam-nhan-s